In [12]:
from path_config import PathConfig
from coverage import calculate_coverage
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
import pandas as pd
import numpy as np

def run_coverage_evaluation(
    tx_lat,
    tx_lon,
    tx_height_m,
    azimuth_deg=0,
    h_bw_deg=65,
    v_bw_deg=10,
    sheet_name="Series Formatted Data",
    pci_target=30
):
    # 1. Dosya yolunu al
    paths = PathConfig()
    dl_path = paths.dl_path

    # 2. Veriyi oku
    df = pd.read_excel(dl_path, sheet_name=sheet_name)

    # 3. Filtreleme
    df_target = df[df["NR_UE_PCI_0"] == pci_target]
    df_target = df_target.dropna(subset=["NR_UE_Nbr_RSRP_0"])

    results = []

    for idx, row in df_target.iterrows():
        rx_lat = row["Latitude"]
        rx_lon = row["Longitude"]

        result = calculate_coverage(
            tx_lat=tx_lat,
            tx_lon=tx_lon,
            rx_lat=rx_lat,
            rx_lon=rx_lon,
            azimuth_deg=azimuth_deg,
            tx_height_m=tx_height_m,
            h_bw_deg=h_bw_deg,
            v_bw_deg=v_bw_deg
        )

        result["Latitude"] = rx_lat
        result["Longitude"] = rx_lon
        result["Time"] = row["Time"]
        result["Actual RSRP"] = row["NR_UE_Nbr_RSRP_0"]

        results.append(result)

    # 4. Birleştir ve sadeleştir
    coverage_df = pd.concat(results, ignore_index=True)
    coverage_df = coverage_df[[
        "Latitude",
        "Longitude",
        "Time",
        "RSRP (dBm)",
        "Actual RSRP"
    ]].rename(columns={"RSRP (dBm)": "Predicted RSRP"})

    # 5. Metrikleri hesapla
    y_true = coverage_df["Actual RSRP"]
    y_pred = coverage_df["Predicted RSRP"]

    mae = mean_absolute_error(y_true, y_pred)
    mse = mean_squared_error(y_true, y_pred)
    rmse = np.sqrt(mse)
    r2 = r2_score(y_true, y_pred)
    
    # MAPE (divide-by-zero olmasın diye epsilon ekliyoruz)
    epsilon = 1e-10
    mape = np.mean(np.abs((y_true - y_pred) / (np.abs(y_true) + epsilon))) * 100

    # 6. Sonuçları yazdır
    print("\n--- KAPSAMA MODELİ DEĞERLENDİRME METRİKLERİ ---")
    print(f"MAE   (Mean Absolute Error)        : {mae:.2f}")
    print(f"MSE   (Mean Squared Error)         : {mse:.2f}")
    print(f"RMSE  (Root Mean Squared Error)    : {rmse:.2f}")
    print(f"MAPE  (Mean Absolute Percentage)   : {mape:.2f}%")
    print(f"R²    (R-squared)                  : {r2:.4f}")
    print("------------------------------------------------\n")

    return coverage_df


In [13]:
if __name__ == "__main__":
    tx_lat = 41.105096
    tx_lon = 29.025006
    tx_height_m = 30

    df_result = run_coverage_evaluation(
        tx_lat=tx_lat,
        tx_lon=tx_lon,
        tx_height_m=tx_height_m
    )

    print(df_result.head())



--- KAPSAMA MODELİ DEĞERLENDİRME METRİKLERİ ---
MAE   (Mean Absolute Error)        : 71.41
MSE   (Mean Squared Error)         : 7999.65
RMSE  (Root Mean Squared Error)    : 89.44
MAPE  (Mean Absolute Percentage)   : 79.92%
R²    (R-squared)                  : -88.2977
------------------------------------------------

   Latitude  Longitude                    Time  Predicted RSRP  Actual RSRP
0  41.10760   29.02257 2025-03-14 12:18:50.499         -206.03        -85.6
1  41.10757   29.02263 2025-03-14 12:18:51.014         -205.75        -87.1
2  41.10756   29.02266 2025-03-14 12:18:51.587         -205.63        -85.6
3  41.10755   29.02268 2025-03-14 12:18:52.016         -205.53        -85.4
4  41.10755   29.02268 2025-03-14 12:18:52.499         -205.53        -84.4
